In [1]:
import os
import json
import pickle
import argparse
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
import tgt
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
)
from collections import Counter
from VAD_chunk import vad_chunk_with_timestamps_v2
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ---------------------------------------------------------------------------
# Model imports (for inlined Wav2Vec2ForCTCAlignment)
# ---------------------------------------------------------------------------
from dataclasses import dataclass
from typing import Optional, Tuple
from transformers import Wav2Vec2ForPreTraining, BertForMaskedLM, BertConfig
from transformers.file_utils import ModelOutput
from transformers.modeling_outputs import MaskedLMOutput


In [2]:
class GumbelQuantizerEMA(nn.Module):
    def __init__(self, input_dim: int, num_vars: int = 320,
                 temp: float = 2.0, decay: float = 0.99):
        super().__init__()
        self.num_vars = num_vars
        self.temp     = temp
        self.decay    = decay

        self.weight_proj = nn.Linear(input_dim, num_vars)

        self.register_buffer("codevectors", torch.randn(num_vars, input_dim))
        self.register_buffer("cluster_size", torch.ones(num_vars))
        self.register_buffer("ema_embed",   torch.randn(num_vars, input_dim))
        nn.init.uniform_(self.codevectors, -1.0, 1.0)

    def forward(self, X: torch.Tensor):
        B, T, D = X.shape
        logits   = self.weight_proj(X)

        if self.training:
            probs = F.gumbel_softmax(logits, tau=self.temp, hard=True)

            with torch.no_grad():
                flat_probs = probs.reshape(-1, self.num_vars)
                flat_X     = X.reshape(-1, D)
                counts     = flat_probs.sum(0)
                embed_sum  = flat_probs.T @ flat_X

                self.cluster_size = (self.decay * self.cluster_size
                                     + (1 - self.decay) * counts)
                self.ema_embed    = (self.decay * self.ema_embed
                                     + (1 - self.decay) * embed_sum)
                self.codevectors  = (self.ema_embed
                                     / self.cluster_size.unsqueeze(1).clamp(min=1e-5))
        else:
            indices = logits.argmax(dim=-1)
            probs   = F.one_hot(indices, self.num_vars).float()

        quantized = torch.matmul(probs, self.codevectors)
        return quantized, probs
from transformers import AutoModel,AutoTokenizer

# =========================
# Corpus-label → IPA mapping (must match training)
# =========================
YOUR_TO_IPA = {
    "a":  "a",  "b":  "b",  "d":  "d",
    "e":  "e",  "f":  "f",  "i":  "i",  "j":  "j",
    "k":  "k",  "l":  "l",  "m":  "m",  "n":  "n",
    "o":  "o",  "p":  "p",  "s":  "s",  "t":  "t",
    "u":  "u",  "v":  "v",  "w":  "w",  "y":  "y",
    "z":  "z",  "ø":  "ø",  "ŋ":  "ŋ",  "ɔ":  "ɔ",
    "ə":  "ə",  "ɛ":  "ɛ",  "ɡ":  "ɡ",  "ɲ":  "ɲ",
    "ʁ":  "ʁ",  "ʃ":  "ʃ",  "ʒ":  "ʒ",
    "dʒ": "ʒ",  "tʃ": "ʃ",  "ts": "s",  "@":  "ə",
    "§":     None,
    "*":     None,
    "[PAD]": None,
    "[UNK]": None,
}


# =========================
# Inference model
# =========================
class Wav2Vec2ForCTC_FS_REC_Inference(nn.Module):

    def __init__(
        self,
        base_model:        Wav2Vec2ForCTC,
        ctc_tokenizer,
        hidden_dim:        int,
        align_temperature: float = 15.0,
        diag_strength:     float = 30.0,
        ahead_strength:    float = 40.0,
    ):
        super().__init__()
        self.wav2vec2          = base_model.wav2vec2
        self.ctc_head          = base_model.lm_head
        self.ctc_tokenizer     = ctc_tokenizer
        self.align_temperature = align_temperature
        self.diag_strength     = diag_strength
        self.ahead_strength    = ahead_strength

        print("Loading XPhoneBERT...")
        self.phoneme_bert   = AutoModel.from_pretrained("vinai/xphonebert-base")
        self.xphonebert_tok = AutoTokenizer.from_pretrained(
            "vinai/xphonebert-base", add_prefix_space=True
        )
        for p in self.phoneme_bert.parameters():
            p.requires_grad = False

        # Must match training exactly
        self.bert_proj = nn.Sequential(
            nn.Linear(768, hidden_dim),
            nn.ReLU(),
        )
        self.fx = nn.Linear(hidden_dim, hidden_dim, bias=False)
        nn.init.eye_(self.fx.weight)

        # ← ADDED: fy fuses Y_ctc and Y_bert
        self.fy = nn.Linear(2 * hidden_dim, hidden_dim, bias=False)

    def _ctc_decode_batch(self, logits):
        pred_ids  = logits.argmax(dim=-1)
        sequences = []
        for seq in pred_ids:
            unique_mask = torch.cat([
                torch.tensor([True], device=seq.device),
                seq[1:] != seq[:-1]
            ])
            collapsed = seq[unique_mask]
            collapsed = collapsed[collapsed != 0]
            if collapsed.numel() == 0:
                collapsed = torch.tensor([1], device=seq.device)
            sequences.append(collapsed)
        return torch.nn.utils.rnn.pad_sequence(
            sequences, batch_first=True, padding_value=0
        )

    def decode_to_ipa(self, labels_clean):
        ipa_sequences = []
        for seq in labels_clean:
            ipa_phones = []
            for token_id in seq:
                tid = token_id.item()
                if tid == 0:
                    break
                corpus_label = self.ctc_tokenizer.convert_ids_to_tokens(tid)
                ipa          = YOUR_TO_IPA.get(corpus_label, None)
                if ipa is not None:
                    ipa_phones.append(ipa)
            if not ipa_phones:
                ipa_phones = ["ə"]
            ipa_sequences.append(ipa_phones)
        return ipa_sequences

    # ── Soft CTC pooling (same as training) ──────────────────────────
    def compute_ctc_embeddings(self, X, logits, labels_clean):
        B, T, D    = X.shape
        ctc_probs  = logits.softmax(dim=-1)          # (B, T, V)
        V          = ctc_probs.shape[-1]             # vocab size
        token_mass = ctc_probs.sum(dim=1).clamp(min=1e-6)
        token_bank = torch.einsum("btd,btv->bvd", X, ctc_probs)
        token_bank = token_bank / token_mass.unsqueeze(-1)
    
        Y_ctc_list = []
        for b in range(B):
            seq = labels_clean[b]
            seq = seq[seq != 0]
            if seq.numel() == 0:
                seq = torch.tensor([1], device=X.device, dtype=torch.long)
    
            # Safety clamp — prevents CUDA index out-of-bounds
            seq = seq.clamp(0, V - 1)
    
            # Debug: print if any clamping actually happened
            if (seq >= V).any():
                print(f"  ⚠ batch {b}: seq contained index >= V={V}, clamping applied")
    
            Y_ctc_list.append(token_bank[b].index_select(0, seq))
    
        return torch.nn.utils.rnn.pad_sequence(
            Y_ctc_list, batch_first=True, padding_value=0.0
        )
    def _bert_encode_chunk(self, ipa_seq: list, device) -> torch.Tensor:
        """
        Run XPhoneBERT on one chunk guaranteed to be <= 512 BPE tokens.
        Aggregates BPE sub-tokens back to phoneme level via word_ids().
        """
        enc    = self.xphonebert_tok(
            [ipa_seq], return_tensors="pt",
            is_split_into_words=True, padding=False,
        ).to(device)
        with torch.no_grad():
            out = self.phoneme_bert(**enc)
        hidden   = out.last_hidden_state[0]
        word_ids = enc.word_ids(batch_index=0)
        embs = []
        for n in range(len(ipa_seq)):
            positions = [i for i, w in enumerate(word_ids) if w == n]
            if positions:
                embs.append(hidden[positions, :].mean(dim=0))
            else:
                embs.append(torch.zeros(768, device=device))
        return torch.stack(embs)   # (N_phones, 768)

    def _bert_forward_single(self, ipa_seq: list, device) -> torch.Tensor:
        """
        Run XPhoneBERT on one phoneme sequence of any length.
        If the BPE token count exceeds 510, splits into chunks using
        binary search on phoneme boundaries — no audio is cut.
        Returns (N_phones, 768).
        """
        MAX_BPE = 510

        # Check BPE length on CPU before any GPU call
        enc_check = self.xphonebert_tok(
            [ipa_seq], is_split_into_words=True, add_special_tokens=False
        )
        n_bpe = len(enc_check["input_ids"][0])

        if n_bpe <= MAX_BPE:
            # Fast path — fits in one BERT call
            return self._bert_encode_chunk(ipa_seq, device)

        # Slow path — sequence too long, chunk at phoneme boundaries
        print(f"  ⚠  Phoneme sequence {len(ipa_seq)} phones "
              f"({n_bpe} BPE tokens) — chunking for XPhoneBERT")

        N_phones    = len(ipa_seq)
        phone_embs  = [None] * N_phones
        chunk_start = 0

        while chunk_start < N_phones:
            # Binary search: find largest chunk starting at chunk_start
            # whose BPE count <= MAX_BPE
            lo, hi = 1, N_phones - chunk_start
            while lo < hi:
                mid = (lo + hi + 1) // 2
                sub = ipa_seq[chunk_start : chunk_start + mid]
                enc = self.xphonebert_tok(
                    [sub], is_split_into_words=True, add_special_tokens=False
                )
                if len(enc["input_ids"][0]) <= MAX_BPE:
                    lo = mid
                else:
                    hi = mid - 1

            chunk_end = chunk_start + lo
            chunk_emb = self._bert_encode_chunk(
                ipa_seq[chunk_start:chunk_end], device
            )   # (chunk_len, 768)

            for i, emb in enumerate(chunk_emb):
                phone_embs[chunk_start + i] = emb

            chunk_start = chunk_end

        return torch.stack(phone_embs)   # (N_phones, 768)

    def compute_bert_embeddings(self, labels_clean, device):
        """
        Returns H_bert: (B, N_max, 768).
        Calls _bert_forward_single for each batch item — handles any length.
        """
        ipa_sequences = self.decode_to_ipa(labels_clean)
        all_embs = []
        for ipa_seq in ipa_sequences:
            if len(ipa_seq) == 0:
                all_embs.append(torch.zeros(1, 768, device=device))
                continue
            all_embs.append(self._bert_forward_single(ipa_seq, device))
        return torch.nn.utils.rnn.pad_sequence(
            all_embs, batch_first=True, padding_value=0.0
        )   # (B, N_max, 768)


    # ── Fused embeddings (same as training) ──────────────────────────
    def compute_phoneme_embeddings(self, X, logits, labels_clean, device):
        B = X.shape[0]

        Y_ctc  = self.compute_ctc_embeddings(X, logits, labels_clean)  # (B, N, D)

        H_bert = self.compute_bert_embeddings(labels_clean, device).to(device)
        Y_bert = self.bert_proj(H_bert)                                 # (B, N, D)

        N      = Y_ctc.shape[1]
        N_bert = Y_bert.shape[1]
        if N_bert >= N:
            Y_bert = Y_bert[:, :N, :]
        else:
            pad    = torch.zeros(B, N - N_bert, Y_ctc.shape[-1],
                                 device=device, dtype=Y_bert.dtype)
            Y_bert = torch.cat([Y_bert, pad], dim=1)

        return self.fy(torch.cat([Y_ctc, Y_bert], dim=-1))             # (B, N, D)

    def compute_alignment(self, X, Y_emb):
        B, T, D = X.shape
        _, N, _ = Y_emb.shape
        X_proj  = self.fx(X)
        X_norm  = F.normalize(X_proj, dim=-1)
        Y_norm  = F.normalize(Y_emb,  dim=-1)
        D_mat   = self.align_temperature * torch.matmul(
            Y_norm, X_norm.transpose(1, 2)
        )
        n_idx = torch.arange(N, device=X.device).float() / max(N - 1, 1)
        t_idx = torch.arange(T, device=X.device).float() / max(T - 1, 1)
        diff          = n_idx.unsqueeze(1) - t_idx.unsqueeze(0)
        gaussian      = -self.diag_strength  * diff ** 2
        ahead_penalty = -self.ahead_strength * torch.clamp(diff, min=0) ** 2
        D_mat         = D_mat + (gaussian + ahead_penalty).unsqueeze(0)
        return torch.softmax(D_mat, dim=1)

    def forward(self, input_values, attention_mask=None):
        outputs      = self.wav2vec2(input_values, attention_mask=attention_mask)
        X            = outputs.last_hidden_state
        logits       = self.ctc_head(X)
        labels_clean = self._ctc_decode_batch(logits)
        Y_emb        = self.compute_phoneme_embeddings(
            X, logits, labels_clean, device=X.device   # ← X and logits now passed
        )
        A = self.compute_alignment(X, Y_emb)
        return {
            "logits":       logits,
            "alignment":    A,
            "labels_clean": labels_clean,
        }

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, Wav2Vec2ForCTC
#xphonebert modified with positional embedding
# =========================
# IPA mapping (must match training exactly)
# =========================
YOUR_TO_IPA = {
    "a":"a",  "b":"b",  "d":"d",  "e":"e",  "f":"f",
    "i":"i",  "j":"j",  "k":"k",  "l":"l",  "m":"m",
    "n":"n",  "o":"o",  "p":"p",  "s":"s",  "t":"t",
    "u":"u",  "v":"v",  "w":"w",  "y":"y",  "z":"z",
    "ø":"ø",  "ŋ":"ŋ",  "ɔ":"ɔ",  "ə":"ə",  "ɛ":"ɛ",
    "ɡ":"ɡ",  "ɲ":"ɲ",  "ʁ":"ʁ",  "ʃ":"ʃ",  "ʒ":"ʒ",
    "dʒ":"ʒ", "tʃ":"ʃ", "ts":"s", "@":"ə",
    "§":None, "*":None, "[PAD]":None, "[UNK]":None,
}


class Wav2Vec2ForCTC_FS_REC_v2_Inference(nn.Module):
    """
    Inference-only class for XPhoneBERT v2 model.

    Matches training model exactly:
      - Detached CTC pooling (X.detach, logits.detach)
      - XPhoneBERT frozen + bert_proj trainable
      - pos_embed: learnable positional encoding added to Y_bert
      - fy: fuses cat([Y_ctc, Y_bert + pos])
      - fx: projects X for alignment
      - align_temperature=20, diag_strength=50, ahead_strength=60

    Dropped (training-only):
      - GumbelQuantizerEMA
      - reconstruction_head
      - mask_features
    """

    def __init__(
        self,
        base_model:        Wav2Vec2ForCTC,
        ctc_tokenizer,
        hidden_dim:        int,
        align_temperature: float = 20.0,
        diag_strength:     float = 50.0,
        ahead_strength:    float = 60.0,
    ):
        super().__init__()

        self.wav2vec2          = base_model.wav2vec2
        self.ctc_head          = base_model.lm_head
        self.ctc_tokenizer     = ctc_tokenizer
        self.align_temperature = align_temperature
        self.diag_strength     = diag_strength
        self.ahead_strength    = ahead_strength

        # ── Alignment projection ──────────────────────────────────────────
        self.fx = nn.Linear(hidden_dim, hidden_dim, bias=False)
        nn.init.eye_(self.fx.weight)

        # ── XPhoneBERT — frozen ───────────────────────────────────────────
        print("Loading XPhoneBERT...")
        self.phoneme_bert   = AutoModel.from_pretrained("vinai/xphonebert-base")
        self.xphonebert_tok = AutoTokenizer.from_pretrained(
            "vinai/xphonebert-base", add_prefix_space=True
        )
        for p in self.phoneme_bert.parameters():
            p.requires_grad = False

        # bert_proj: 768 → hidden_dim (weights loaded from checkpoint)
        self.bert_proj = nn.Sequential(
            nn.Linear(768, hidden_dim),
            nn.ReLU(),
        )

        # ── Positional encoding (NEW in v2) ───────────────────────────────
        self.pos_embed = nn.Embedding(512, hidden_dim)
        nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.01)

        # ── Fusion: cat([Y_ctc, Y_bert + pos]) → hidden_dim ──────────────
        self.fy = nn.Linear(2 * hidden_dim, hidden_dim, bias=False)

    # ------------------------------------------------------------------
    # CTC greedy decode
    # ------------------------------------------------------------------
    def _ctc_decode_batch(self, logits: torch.Tensor) -> torch.Tensor:
        pred_ids  = logits.argmax(dim=-1)
        sequences = []
        for seq in pred_ids:
            unique_mask = torch.cat([
                torch.tensor([True], device=seq.device),
                seq[1:] != seq[:-1]
            ])
            collapsed = seq[unique_mask]
            collapsed = collapsed[collapsed != 0]
            if collapsed.numel() == 0:
                collapsed = torch.tensor([1], device=seq.device)
            sequences.append(collapsed)
        return torch.nn.utils.rnn.pad_sequence(
            sequences, batch_first=True, padding_value=0
        )

    # ------------------------------------------------------------------
    # IPA conversion
    # ------------------------------------------------------------------
    def decode_to_ipa(self, labels_clean: torch.Tensor) -> list:
        ipa_sequences = []
        for seq in labels_clean:
            ipa_phones = []
            for token_id in seq:
                tid = token_id.item()
                if tid == 0:
                    break
                corpus_label = self.ctc_tokenizer.convert_ids_to_tokens(tid)
                ipa          = YOUR_TO_IPA.get(corpus_label, None)
                if ipa is not None:
                    ipa_phones.append(ipa)
            if not ipa_phones:
                ipa_phones = ["ə"]
            ipa_sequences.append(ipa_phones)
        return ipa_sequences

    # ------------------------------------------------------------------
    # Detached CTC pooling
    # ------------------------------------------------------------------
    def compute_ctc_embeddings(self, X, logits, labels_clean):
        B, T, D    = X.shape
        V          = logits.shape[-1]
        ctc_probs  = logits.softmax(dim=-1)
        token_mass = ctc_probs.sum(dim=1).clamp(min=1e-6)
        token_bank = torch.einsum("btd,btv->bvd", X, ctc_probs)
        token_bank = token_bank / token_mass.unsqueeze(-1)
        Y_ctc_list = []
        for b in range(B):
            seq = labels_clean[b]
            seq = seq[seq != 0]
            if seq.numel() == 0:
                seq = torch.tensor([1], device=X.device, dtype=torch.long)
            seq = seq.clamp(0, V - 1)
            Y_ctc_list.append(token_bank[b].index_select(0, seq))
        return torch.nn.utils.rnn.pad_sequence(
            Y_ctc_list, batch_first=True, padding_value=0.0
        )

    # ------------------------------------------------------------------
    # XPhoneBERT with BPE aggregation + chunking for long sequences
    # ------------------------------------------------------------------
    def _bert_encode_chunk(self, ipa_seq: list, device) -> torch.Tensor:
        enc    = self.xphonebert_tok(
            [ipa_seq], return_tensors="pt",
            is_split_into_words=True, padding=False,
        ).to(device)
        with torch.no_grad():
            out = self.phoneme_bert(**enc)
        hidden   = out.last_hidden_state[0]
        word_ids = enc.word_ids(batch_index=0)
        embs = []
        for n in range(len(ipa_seq)):
            positions = [i for i, w in enumerate(word_ids) if w == n]
            if positions:
                embs.append(hidden[positions, :].mean(dim=0))
            else:
                embs.append(torch.zeros(768, device=device))
        return torch.stack(embs)

    def _bert_forward_single(self, ipa_seq: list, device) -> torch.Tensor:
        MAX_BPE   = 510
        enc_check = self.xphonebert_tok(
            [ipa_seq], is_split_into_words=True, add_special_tokens=False
        )
        if len(enc_check["input_ids"][0]) <= MAX_BPE:
            return self._bert_encode_chunk(ipa_seq, device)
        # Binary search chunking for sequences > 510 BPE tokens
        N_phones    = len(ipa_seq)
        phone_embs  = [None] * N_phones
        chunk_start = 0
        while chunk_start < N_phones:
            lo, hi = 1, N_phones - chunk_start
            while lo < hi:
                mid = (lo + hi + 1) // 2
                sub = ipa_seq[chunk_start : chunk_start + mid]
                enc = self.xphonebert_tok(
                    [sub], is_split_into_words=True, add_special_tokens=False
                )
                if len(enc["input_ids"][0]) <= MAX_BPE:
                    lo = mid
                else:
                    hi = mid - 1
            chunk_end = chunk_start + lo
            chunk_emb = self._bert_encode_chunk(
                ipa_seq[chunk_start:chunk_end], device
            )
            for i, emb in enumerate(chunk_emb):
                phone_embs[chunk_start + i] = emb
            chunk_start = chunk_end
        return torch.stack(phone_embs)

    def compute_bert_embeddings(self, labels_clean, device):
        ipa_sequences = self.decode_to_ipa(labels_clean)
        all_embs = []
        for ipa_seq in ipa_sequences:
            if len(ipa_seq) == 0:
                all_embs.append(torch.zeros(1, 768, device=device))
                continue
            all_embs.append(self._bert_forward_single(ipa_seq, device))
        return torch.nn.utils.rnn.pad_sequence(
            all_embs, batch_first=True, padding_value=0.0
        )

    # ------------------------------------------------------------------
    # Fused phoneme embeddings with positional encoding
    # ------------------------------------------------------------------
    def compute_phoneme_embeddings(self, X, logits, labels_clean, device):
        B = X.shape[0]

        # Detached CTC pooling — no gradient to encoder
        Y_ctc  = self.compute_ctc_embeddings(
            X.detach(), logits.detach(), labels_clean
        )

        # XPhoneBERT contextual embeddings
        H_bert = self.compute_bert_embeddings(labels_clean, device).to(device)
        Y_bert = self.bert_proj(H_bert)

        # Align N dimension
        N      = Y_ctc.shape[1]
        N_bert = Y_bert.shape[1]
        if N_bert >= N:
            Y_bert = Y_bert[:, :N, :]
        else:
            pad    = torch.zeros(B, N - N_bert, Y_ctc.shape[-1],
                                 device=device, dtype=Y_bert.dtype)
            Y_bert = torch.cat([Y_bert, pad], dim=1)

        # Positional encoding — makes each position unique
        positions = torch.arange(N, device=device).clamp(max=511)
        pos       = self.pos_embed(positions).unsqueeze(0)
        Y_bert    = Y_bert + pos

        # Fuse
        return self.fy(torch.cat([Y_ctc, Y_bert], dim=-1))

    # ------------------------------------------------------------------
    # Alignment
    # ------------------------------------------------------------------
    def compute_alignment(self, X: torch.Tensor, Y_emb: torch.Tensor) -> torch.Tensor:
        B, T, D = X.shape
        _, N, _ = Y_emb.shape
        X_proj  = self.fx(X)
        X_norm  = F.normalize(X_proj, dim=-1)
        Y_norm  = F.normalize(Y_emb,  dim=-1)
        D_mat   = self.align_temperature * torch.matmul(
            Y_norm, X_norm.transpose(1, 2)
        )
        n_idx = torch.arange(N, device=X.device).float() / max(N - 1, 1)
        t_idx = torch.arange(T, device=X.device).float() / max(T - 1, 1)
        diff          = n_idx.unsqueeze(1) - t_idx.unsqueeze(0)
        gaussian      = -self.diag_strength  * diff ** 2
        ahead_penalty = -self.ahead_strength * torch.clamp(diff, min=0) ** 2
        D_mat         = D_mat + (gaussian + ahead_penalty).unsqueeze(0)
        return torch.softmax(D_mat, dim=1)

    # ------------------------------------------------------------------
    # Forward
    # ------------------------------------------------------------------
    def forward(self, input_values, attention_mask=None):
        outputs      = self.wav2vec2(input_values, attention_mask=attention_mask)
        X            = outputs.last_hidden_state
        logits       = self.ctc_head(X)
        labels_clean = self._ctc_decode_batch(logits)
        Y_emb        = self.compute_phoneme_embeddings(
            X, logits, labels_clean, device=X.device
        )
        A = self.compute_alignment(X, Y_emb)
        return {
            "logits":       logits,
            "alignment":    A,
            "labels_clean": labels_clean,
        }

In [17]:
#softpooling only

class Wav2Vec2ForCTC_FS_REC_Inference(nn.Module):
    """
    Inference class for soft CTC pooling model (no XPhoneBERT).
    """

    def __init__(
        self,
        base_model:        Wav2Vec2ForCTC,
        ctc_tokenizer,
        hidden_dim:        int,
        align_temperature: float = 15.0,
        diag_strength:     float = 30.0,
        ahead_strength:    float = 40.0,
    ):
        super().__init__()
        self.wav2vec2          = base_model.wav2vec2
        self.ctc_head          = base_model.lm_head
        self.ctc_tokenizer     = ctc_tokenizer
        self.align_temperature = align_temperature
        self.diag_strength     = diag_strength
        self.ahead_strength    = ahead_strength

        self.fx = nn.Linear(hidden_dim, hidden_dim, bias=False)
        nn.init.eye_(self.fx.weight)
        self.fy = nn.Linear(hidden_dim, hidden_dim, bias=False)
        nn.init.eye_(self.fy.weight)

        # No phoneme_bert, no xphonebert_tok, no bert_proj

    def _ctc_decode_batch(self, logits: torch.Tensor) -> torch.Tensor:
        pred_ids  = logits.argmax(dim=-1)
        sequences = []
        for seq in pred_ids:
            unique_mask = torch.cat([
                torch.tensor([True], device=seq.device),
                seq[1:] != seq[:-1]
            ])
            collapsed = seq[unique_mask]
            collapsed = collapsed[collapsed != 0]
            if collapsed.numel() == 0:
                collapsed = torch.tensor([1], device=seq.device)
            sequences.append(collapsed)
        return torch.nn.utils.rnn.pad_sequence(
            sequences, batch_first=True, padding_value=0
        )

    def compute_phoneme_embeddings(self, X, logits, labels_clean):
        """
        Soft CTC pooling only — no XPhoneBERT.
        token_bank[v] = weighted average of X frames where P(v|t) is high.
        """
        B, T, D    = X.shape
        V          = logits.shape[-1]
        ctc_probs  = logits.softmax(dim=-1)                           # (B, T, V)
        token_mass = ctc_probs.sum(dim=1).clamp(min=1e-6)             # (B, V)
        token_bank = torch.einsum("btd,btv->bvd", X, ctc_probs)       # (B, V, D)
        token_bank = token_bank / token_mass.unsqueeze(-1)

        Y_list = []
        for b in range(B):
            seq = labels_clean[b]
            seq = seq[seq != 0]
            if seq.numel() == 0:
                seq = torch.tensor([1], device=X.device, dtype=torch.long)
            seq = seq.clamp(0, V - 1)   # safety clamp
            Y_list.append(token_bank[b].index_select(0, seq))

        Y_ctc = torch.nn.utils.rnn.pad_sequence(
            Y_list, batch_first=True, padding_value=0.0
        )   # (B, N, D)

        return self.fy(Y_ctc)   # (B, N, D)

    def compute_alignment(self, X: torch.Tensor, Y_emb: torch.Tensor) -> torch.Tensor:
        B, T, D = X.shape
        _, N, _ = Y_emb.shape
        X_proj  = self.fx(X)
        X_norm  = F.normalize(X_proj, dim=-1)
        Y_norm  = F.normalize(Y_emb,  dim=-1)
        D_mat   = self.align_temperature * torch.matmul(
            Y_norm, X_norm.transpose(1, 2)
        )
        n_idx = torch.arange(N, device=X.device).float() / max(N - 1, 1)
        t_idx = torch.arange(T, device=X.device).float() / max(T - 1, 1)
        diff          = n_idx.unsqueeze(1) - t_idx.unsqueeze(0)
        gaussian      = -self.diag_strength  * diff ** 2
        ahead_penalty = -self.ahead_strength * torch.clamp(diff, min=0) ** 2
        D_mat         = D_mat + (gaussian + ahead_penalty).unsqueeze(0)
        return torch.softmax(D_mat, dim=1)

    def forward(self, input_values, attention_mask=None):
        outputs      = self.wav2vec2(input_values, attention_mask=attention_mask)
        X            = outputs.last_hidden_state
        logits       = self.ctc_head(X)
        labels_clean = self._ctc_decode_batch(logits)
        Y_emb        = self.compute_phoneme_embeddings(X.detach(), logits.detach(), labels_clean)
        A            = self.compute_alignment(X, Y_emb)
        return {
            "logits":       logits,
            "alignment":    A,
            "labels_clean": labels_clean,
        }

In [4]:
sampa_to_api_single = {'a': 'a', 'e': 'e','i': 'i','o': 'o','u': 'u','y': 'y','2': 'ø','9': '9','@': 'ə','E': 'ɛ','O': 'ɔ','a~': '@', 'e~': '5', '9~': '1',    
    'o~': '§', 'b': 'b','d': 'd','f': 'f','g': 'ɡ',
    'k': 'k','l': 'l','m': 'm','n': 'n','n=':'n','p': 'p','t': 't','v': 'v','w': 'w','z': 'z','j': 'j','R': 'ʁ','N': 'ŋ','H': 'ɥ','J': 'ɲ','S': 'ʃ','Z': 'ʒ','Z=': 'ʒ',
    's': 's',
    'm=': 'm',
    '_': '_',
    'spn': 'spn',
    'unk': 'spn',
    "%":'spn',
    "?":"spn",
    "0":"spn"
}
with open("vocab_w2vCTC-woSIL.json") as f:
    vocab = json.load(f)
hyp_to_ref = {
    **sampa_to_api_single,
    '@': '@',    # override: vocab '@' is the nasal vowel, not schwa
    'ts': 's',    # ADD — model sometimes outputs affricate, collapse to /s/
    'tʃ': 'ʃ'
}
def normalise_hyp_token(tok: str) -> str:
    """Convert a raw tokenizer token to the REF label space."""
    return hyp_to_ref.get(tok, tok)   # passthrough if not in map
def map_ref_to_api_single(ph, mapping):
    if ph in mapping:
        return mapping[ph]
    if ph is not None and ph != "":
        return ph
    return None

In [5]:
import soundfile as sf

def get_ref_intervals(clean_ref, audio_path, eps=0.1):
    if not clean_ref:
        return []
    audio, sr = sf.read(audio_path)
    audio_duration = len(audio) / sr
    ref_first      = clean_ref[0]["start"]
    ref_last       = clean_ref[-1]["end"]
    end_overshoot  = ref_last - audio_duration
    ref_span       = ref_last - ref_first

    needs_correction = False
    reason = ""

    # Case 1: ref clearly extends past audio
    if end_overshoot > eps:
        needs_correction = True
        reason = "end_overshoot"
    # Case 2: ref starts well after 0 but its span fits within the audio
    elif ref_first > 0.3 and ref_span <= audio_duration + eps:
        needs_correction = True
        reason = "ref_first only"

    if needs_correction:
        offset = ref_first
        print(f"  Session offset corrected ({reason}): "
              f"ref_first={ref_first:.3f}s, ref_last={ref_last:.3f}s, "
              f"audio={audio_duration:.3f}s, end_overshoot={end_overshoot:+.3f}s, "
              f"offset={offset:.3f}s")
        return [
            {
                "phoneme": item["phoneme"],
                "start":   max(0.0, round(item["start"] - offset, 6)),
                "end":     max(0.0, round(item["end"]   - offset, 6)),
            }
            for item in clean_ref
        ]
    else:
        return list(clean_ref)
from librosa.sequence import dtw
from itertools import groupby
def read_textgrid(tg_path, tier_name="phone"):
    """
    Returns list of {'phoneme': str, 'start': float, 'end': float}.
    Skips silence intervals (empty label, SIL, sil, sp).
    """
    tg        = tgt.io.read_textgrid(tg_path)
    tier      = tg.get_tier_by_name(tier_name)
    silence_labels = {"", "SIL", "sil", "spn", "SP", "<SIL>", "_",
                      "0", "fe~", "sjo~", "Ra~"}
    intervals = []
    for iv in tier.intervals:
        label = iv.text.strip()
        if not label or label in silence_labels:
            continue
        phoneme = map_ref_to_api_single(label, sampa_to_api_single)
        if phoneme is None or phoneme in silence_labels:
            continue
        intervals.append({
            "phoneme": phoneme,
            "start":   round(iv.start_time, 6),
            "end":     round(iv.end_time,   6),
        })
    return intervals






def _dtw_align_energy(A_TN: np.ndarray) -> np.ndarray:
    """Original-charsiu-style forced alignment via DTW on the attention energy.

    Mirrors src/utils.py:forced_align in upstream charsiu:
        D, wp = dtw(C=-softmax(out.logits)[:, target_phone_ids],
                    step_sizes_sigma=[[1, 1], [1, 0]])
    Step sizes [1, 1] (advance frame and phone) and [1, 0] (advance frame,
    stay on phone) — no skip, no backtracking, every frame consumed exactly
    once, no blank states. Each phone in the target sequence gets at least
    one frame.

    A_TN: [T, N] attention energy already sliced to the N collapsed phones.
    Returns: [T] int array, per-frame phone index in 0..N-1 as chosen by DTW.
    """
    T, N = A_TN.shape
    if N == 0:
        return np.zeros(T, dtype=np.int32)
    if N == 1:
        return np.zeros(T, dtype=np.int32)

    # softmax over phones → P(phone | frame); negate so DTW minimises.
    e = A_TN.astype(np.float64)
    e -= e.max(axis=1, keepdims=True)
    pe = np.exp(e)
    cost = pe / pe.sum(axis=1, keepdims=True)            # [T, N]

    D, wp = dtw(C=-cost,
                step_sizes_sigma=np.array([[1, 1], [1, 0]]))

    # librosa returns the path in reverse (end → start). With these step
    # sizes every frame appears at most once; take the max phone index per
    # frame as a safety guard (upstream does the same).
    per_frame = np.full(T, -1, dtype=np.int32)
    for fr, ph in wp:
        if 0 <= fr < T and per_frame[fr] < ph:
            per_frame[fr] = ph

    # Forward-fill any unvisited frames (shouldn't happen with these step
    # sizes, but be defensive).
    last = 0
    for t in range(T):
        if per_frame[t] < 0:
            per_frame[t] = last
        else:
            last = per_frame[t]
    return per_frame


def extract_intervals_forced(logits, A, tokenizer, duration_sec: float,
                              frame_shift: float = 0.02,
                              plot_path: str = None,
                              plot_title: str = "",
                              plot_time_offset: float = 0.0) -> list:
    """Build (start, end, phone) intervals for one chunk using upstream
    charsiu's strategy: DTW alignment on the (softmaxed) attention energy
    against the CTC-collapsed phone sequence, then run-length grouping of
    per-frame phone indices into intervals via seq2duration-style counting.

    Times here are *chunk-local* (frame_index × frame_shift). The caller is
    responsible for adding chunk.pad_start to turn them into absolute times.
    Each chunk is treated as an independent audio file — full pipeline run
    end-to-end on its frames, no cross-chunk coupling in the alignment.
    """
    blank_id = tokenizer.pad_token_id
    skip = {"[PAD]", "[UNK]", "spn", "", None}

    pred_ids = logits[0].argmax(dim=-1).tolist()
    collapsed = []
    prev = None
    for tok in pred_ids:
        if tok != prev:
            if tok != blank_id:
                collapsed.append(tok)
        prev = tok

    # Diagnostic.
    non_blank = sum(1 for t in pred_ids if t != blank_id)
    skipped_unk_spn = sum(
        1 for c in collapsed
        if normalise_hyp_token(tokenizer.convert_ids_to_tokens(c)) in skip
    )
    print(
        f"  [chunk] raw_frames={len(pred_ids)}  non_blank={non_blank}  "
        f"collapsed={len(collapsed)}  skipped_unk_spn={skipped_unk_spn}  "
        f"kept={len(collapsed) - skipped_unk_spn}"
    )

    if not collapsed:
        return []

    N = len(collapsed)
    T = logits.shape[1]

    # CTC-blank silence mask. Mirrors the `sil_mask` / `nonsil_idx` step in
    # src/Charsiu.py:143 — DTW should only see speech frames, otherwise the
    # forced start at (frame=0, phone=0) and the [[1,1],[1,0]] step sizes
    # make the first phone absorb any leading pad silence (= the "stuck on
    # phone 0 for X frames" pattern visible in the chunk plots). The CTC
    # head's argmax is our silence detector: frames whose argmax is [PAD]
    # are blank/silence.
    pred_arr  = np.asarray(pred_ids, dtype=np.int64)
    is_speech = pred_arr != blank_id
    speech_idx = np.flatnonzero(is_speech)

    A_TN = A[0].cpu().float().numpy().T           # [T, N]

    if speech_idx.size == 0:
        # CTC saw only blank in this chunk — nothing to align.
        return []

    # DTW runs on the non-silence frames only; result has length T_speech.
    A_TN_speech = A_TN[speech_idx]
    per_frame_phone_speech = _dtw_align_energy(A_TN_speech)

    # Lift back to full-length [T] frame assignment. Silence frames get the
    # sentinel -1 so the groupby below can recognise them as "_" runs and
    # skip them when emitting intervals (their timestamps still consume the
    # counter, so subsequent phone intervals get their correct absolute
    # start time = first speech frame, not chunk frame 0).
    per_frame_phone = np.full(T, -1, dtype=np.int32)
    per_frame_phone[speech_idx] = per_frame_phone_speech

    # Optional per-chunk plot of the energy heatmap + DTW path.
    if plot_path is not None:
        plot_chunk_alignment(
            A_TN=A_TN,
            per_frame_phone=per_frame_phone,
            phone_tokens=collapsed,
            tokenizer=tokenizer,
            output_path=plot_path,
            frame_shift=frame_shift,
            title=plot_title,
            time_offset=plot_time_offset,
        )

    # seq2duration: run-length encode the per-frame phone index sequence.
    # Silence runs (phone_idx == -1) advance the counter but produce no
    # interval — mirrors upstream's _merge_silence by simply omitting silence
    # from the hyp output (the user wants silence dropped on both sides).
    intervals = []
    counter = 0
    for phone_idx, group in groupby(per_frame_phone.tolist()):
        length = sum(1 for _ in group)
        if phone_idx == -1:
            counter += length
            continue
        raw_tok = tokenizer.convert_ids_to_tokens(int(collapsed[phone_idx]))
        phoneme_str = normalise_hyp_token(raw_tok)
        if phoneme_str not in skip:
            intervals.append({
                "phoneme": phoneme_str,
                "start":   round(counter * frame_shift, 6),
                "end":     round((counter + length) * frame_shift, 6),
            })
        counter += length

    return intervals


In [6]:
import numpy as np
from VAD_chunk import *
def intervals_to_frame_mask(intervals, T, frame_shift):
    """
    intervals: list of {"start": float, "end": float}
    returns: boolean mask of shape [T], True = keep frame
    """
    mask = np.zeros(T, dtype=bool)
    frame_centers = (np.arange(T) + 0.5) * frame_shift

    for iv in intervals:
        s = iv["start"]
        e = iv["end"]
        mask |= (frame_centers >= s) & (frame_centers < e)

    return mask
import numpy as np
from itertools import groupby

def extract_intervals_forced_vad(
    logits,
    A,
    tokenizer,
    duration_sec: float,
    chunk: dict = None,
    frame_shift: float = 0.02,
    plot_path: str = None,
    plot_title: str = "",
    plot_time_offset: float = 0.0,
) -> list:
    """
    Build (start, end, phoneme) intervals for one chunk.

    Uses VAD silence regions inside the chunk to mask out silent frames,
    then runs DTW on the remaining speech frames only.
    """
    skip = {"[PAD]", "[UNK]", "spn", "", None}

    # --- CTC argmax only to get the collapsed phone sequence ---
    pred_ids = logits[0].argmax(dim=-1).tolist()
    collapsed = []
    prev = None
    for tok in pred_ids:
        if tok != prev:
            if tok != tokenizer.pad_token_id:
                collapsed.append(tok)
        prev = tok

    if not collapsed:
        return []

    N = len(collapsed)
    T = logits.shape[1]

    # --- Build VAD-based speech mask inside the chunk ---
    # If no chunk / no silence info is provided, keep all frames.
    speech_mask = np.ones(T, dtype=bool)

    if chunk is not None:
        silence_regions = chunk.get("silence_regions", [])
        frame_centers = (np.arange(T) + 0.5) * frame_shift

        for sil in silence_regions:
            s = sil["start"] - chunk["pad_start"]  # convert absolute -> chunk-local
            e = sil["end"]   - chunk["pad_start"]
            speech_mask &= ~((frame_centers >= s) & (frame_centers < e))

    speech_idx = np.flatnonzero(speech_mask)

    if speech_idx.size == 0:
        return []

    # --- DTW on speech frames only ---
    A_TN = A[0].cpu().float().numpy().T          # [T, N]
    A_TN_speech = A_TN[speech_idx]
    per_frame_phone_speech = _dtw_align_energy(A_TN_speech)

    # --- Lift back to full-length frame labels ---
    per_frame_phone = np.full(T, -1, dtype=np.int32)
    per_frame_phone[speech_idx] = per_frame_phone_speech

    # --- Optional plot ---
    if plot_path is not None:
        plot_chunk_alignment(
            A_TN=A_TN,
            per_frame_phone=per_frame_phone,
            phone_tokens=collapsed,
            tokenizer=tokenizer,
            output_path=plot_path,
            frame_shift=frame_shift,
            title=plot_title,
            time_offset=plot_time_offset,
        )

    # --- Run-length grouping into intervals ---
    intervals = []
    counter = 0

    for phone_idx, group in groupby(per_frame_phone.tolist()):
        length = sum(1 for _ in group)

        if phone_idx == -1:
            counter += length
            continue

        raw_tok = tokenizer.convert_ids_to_tokens(int(collapsed[phone_idx]))
        phoneme_str = normalise_hyp_token(raw_tok)

        if phoneme_str not in skip:
            intervals.append({
                "phoneme": phoneme_str,
                "start": round(counter * frame_shift, 6),
                "end": round((counter + length) * frame_shift, 6),
            })

        counter += length

    return intervals

In [7]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def visualize_alignment_with_dtw(entry, fname, tokenizer,
                                 out_dir="results/alignment_plots",
                                 max_phones=80):
    """
    Plot alignment heatmap for each chunk + the FINAL DTW path used by
    extract_intervals_forced().
    """
    os.makedirs(out_dir, exist_ok=True)

    a_chunks = entry.get("A_chunks", [])
    logits_chunks = entry.get("logits_chunks", [])

    if not a_chunks:
        print(f"  No A matrix stored for {fname}")
        return
    if not logits_chunks:
        print(f"  No logits stored for {fname}")
        return

    blank_id = tokenizer.pad_token_id

    for chunk_idx, (A, logits) in enumerate(zip(a_chunks, logits_chunks)):
        # A is typically stored as [N, T]
        A_np = A.numpy() if torch.is_tensor(A) else np.asarray(A)
        if A_np.ndim != 2:
            print(f"  [SKIP] chunk {chunk_idx}: bad A shape {A_np.shape}")
            continue

        N, T = A_np.shape
        N_plot = min(N, max_phones)
        A_plot = A_np[:N_plot, :]

        # Final DTW path comes from the same logic as extract_intervals_forced
        pred_ids = logits[0].argmax(dim=-1).detach().cpu().numpy()
        is_speech = pred_ids != blank_id
        speech_idx = np.flatnonzero(is_speech)

        if speech_idx.size == 0:
            print(f"  [SKIP] chunk {chunk_idx}: no speech frames")
            continue

        # DTW works on speech frames only
        A_TN = A_np.T                      # [T, N]
        A_TN_speech = A_TN[speech_idx]     # [T_speech, N]
        per_frame_phone_speech = _dtw_align_energy(A_TN_speech)

        # Put it back to full chunk length, silence = NaN
        per_frame_phone = np.full(T, np.nan, dtype=np.float32)
        per_frame_phone[speech_idx] = per_frame_phone_speech

        fig = plt.figure(figsize=(min(T * 0.05, 24), 6))
        gs = gridspec.GridSpec(2, 1, height_ratios=[4, 1], hspace=0.08)

        # Top: heatmap + DTW path
        ax_heat = fig.add_subplot(gs[0])
        ax_heat.imshow(
            A_plot,
            aspect="auto",
            origin="upper",
            cmap="Blues",
            interpolation="nearest",
        )

        x = np.arange(T)[~np.isnan(per_frame_phone)]
        y = per_frame_phone[~np.isnan(per_frame_phone)]
        ax_heat.plot(x, y, color="red", lw=1.5, label="DTW path")

        ax_heat.set_ylabel("Phone")
        ax_heat.set_xticks([])
        ax_heat.set_title(
            f"{fname}  chunk {chunk_idx+1}/{len(a_chunks)}  (N={N}, T={T})",
            fontsize=9,
        )
        ax_heat.legend(fontsize=7, loc="upper left")

        # Bottom: DTW path as a step-like sequence
        ax_path = fig.add_subplot(gs[1], sharex=ax_heat)
        ax_path.plot(per_frame_phone, color="darkorange", lw=1.2, label="DTW per-frame phone")
        ax_path.set_xlabel("Frame")
        ax_path.set_ylabel("Ph idx")
        ax_path.set_ylim(-1, N)
        ax_path.legend(fontsize=7, loc="upper left")

        plt.tight_layout()
        save_path = os.path.join(out_dir, f"{fname}_chunk{chunk_idx+1:02d}.png")
        fig.savefig(save_path, dpi=120, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved: {save_path}")

In [8]:
from metrics import *
def prepare_alignment_store(results):
    """
    Convert the inference-loop `results` dict into the format expected by
    metrics() / metrics_rhap().

    Adds:
      - ref_seq:  list of ref phoneme labels (from ref_intervals)
      - hyp_seq:  list of hyp phoneme labels (from hyp_intervals)
      - style:    'M' (monologue) or 'D' (dialogue), inferred from filename
                  (Rhap-M*** vs Rhap-D***). Only needed for metrics_rhap.
    """
    alignment_store = {}
    for filename, data in results.items():
        ref_intervals = data["ref_intervals"]
        hyp_intervals = data["hyp_intervals"]

        # Infer style from filename — Rhap-specific naming
        parts  = filename.split("-")
        prefix = parts[1] if len(parts) > 1 else ""
        if   prefix.startswith("M"): style = "M"
        elif prefix.startswith("D"): style = "D"
        else:                         style = "unknown"

        alignment_store[filename] = {
            "ref_intervals": ref_intervals,
            "hyp_intervals": hyp_intervals,
            "ref_seq":       [iv["phoneme"] for iv in ref_intervals],
            "hyp_seq":       [iv["phoneme"] for iv in hyp_intervals],
            "style":         style,
        }
    return alignment_store

In [9]:
from itertools import groupby
import numpy as np

def extract_intervals_ctc(
    logits,
    tokenizer,
    frame_shift=0.02,
):
    """
    Pure CTC frame-run alignment.
    Returns chunk-local phoneme intervals.
    """

    blank_id = tokenizer.pad_token_id
    skip = {"[PAD]", "[UNK]", "spn", "", None}

    pred_ids = logits[0].argmax(dim=-1).tolist()

    intervals = []
    counter = 0

    for tok, group in groupby(pred_ids):
        length = sum(1 for _ in group)

        # skip blanks but keep time advancing
        if tok == blank_id:
            counter += length
            continue

        phoneme = normalise_hyp_token(
            tokenizer.convert_ids_to_tokens(tok)
        )

        if phoneme not in skip:
            intervals.append({
                "phoneme": phoneme,
                "start": round(counter * frame_shift, 6),
                "end": round((counter + length) * frame_shift, 6),
            })

        counter += length

    return intervals

In [11]:
from transformers import Wav2Vec2Config, BertConfig, Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor
audio_dir = "/vol/corpora/Rhapsodie/wav16k_corrected/"
textgrid_dir = "/vol/corpora/Rhapsodie/TextGrids-fev2013/"
tier = "phone"
device = "cuda"
align_temperature = 20.0
diag_strength = 50.0
ahead_strength= 60.0
#align_temperature = 15.0,
#diag_strength = 30.0,
#ahead_strength = 40.0,
with open("vocab_w2vCTC-woSIL.json") as f:
    vocab = json.load(f)
tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file="/vol/experiments3/imbenamor/TAPAS-FRAIS/alignment_models/vocab_w2vCTC-woSIL.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

ctc_checkpoint = "results/w2vCTC/checkpoint-23430"
# ── Model ─────────────────────────────────────────────────────────────
print(f"Loading CTC architecture from {ctc_checkpoint}...")
base_model = Wav2Vec2ForCTC.from_pretrained(
    ctc_checkpoint,
    ctc_loss_reduction = "mean",
    ctc_zero_infinity  = True,
    pad_token_id       = tokenizer.pad_token_id,
    vocab_size         = len(tokenizer),
)
model = Wav2Vec2ForCTC_FS_REC_v2_Inference(
        base_model        = base_model,
        ctc_tokenizer     = tokenizer,
        hidden_dim        = base_model.config.hidden_size,
        align_temperature = align_temperature,
        diag_strength     = diag_strength,
        ahead_strength    = ahead_strength,
    )
model.wav2vec2.feature_extractor._freeze_parameters()


Loading CTC architecture from results/w2vCTC/checkpoint-23430...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Loading XPhoneBERT...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/xphonebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
len(tokenizer)

42

In [25]:
def vad_chunk_with_timestamps_v3(
    wav,
    sampling_rate=16000,
    max_chunk_duration=10.0,
    max_pause_duration=1.5,
    overlap=0.5,
    min_chunk_duration=2.0,
):
    """
    wav: torch.Tensor (1D, 16kHz)
 
    returns: list of dicts with:
        start/end         -> true speech span of the chunk
        pad_start/pad_end -> padded boundaries used for audio extraction
        speech_regions    -> list of speech intervals inside the chunk
        silence_regions   -> list of silence gaps inside the chunk
 
    Final chunks are kept within [min_chunk_duration, max_chunk_duration]
    whenever possible. Only unfixable case: the whole audio yields a
    single chunk shorter than min_chunk_duration (no neighbour to merge).
    """
    vad = rVADfast()
    vad_labels, vad_timestamps = vad(wav, sampling_rate)
    speech_ts = vad_to_speech_ts(vad_labels, vad_timestamps, sampling_rate)
 
    # Convert VAD timestamps from samples to seconds
    split_ts = []
    for seg in speech_ts:
        seg_start = seg["start"] / sampling_rate
        seg_end = seg["end"] / sampling_rate
        split_ts.append({"start": seg_start, "end": seg_end})
    def _split_long_segment(seg):
        duration = seg["end"] - seg["start"]
        if duration <= max_chunk_duration:
            return [seg]
        n_pieces = math.ceil(duration / max_chunk_duration)
        piece_dur = duration / n_pieces  # even split, each <= max_chunk_duration
        pieces = []
        for i in range(n_pieces):
            p_start = seg["start"] + i * piece_dur
            p_end = seg["end"] if i == n_pieces - 1 else seg["start"] + (i + 1) * piece_dur
            pieces.append({"start": p_start, "end": p_end})
        return pieces

    split_ts = [p for seg in split_ts for p in _split_long_segment(seg)]
    # ── Merge split_ts segments, enforcing min/max chunk duration ────────
    # A chunk keeps growing while EITHER:
    #   • it is still below min_chunk_duration   (force-grow), OR
    #   • the pause to the next segment is short (normal merging)
    # …provided the result still fits under max_chunk_duration.
    chunks = []
    chunk_start = None
    chunk_end = None
    chunk_speech_regions = []
 
    for seg in split_ts:
        seg_start = seg["start"]
        seg_end = seg["end"]
 
        if chunk_start is None:
            chunk_start = seg_start
            chunk_end = seg_end
            chunk_speech_regions = [{"start": seg_start, "end": seg_end}]
            continue
 
        current_duration = chunk_end - chunk_start
        pause = seg_start - chunk_end
        proposed_duration = seg_end - chunk_start
 
        below_min = current_duration < min_chunk_duration
        short_pause = pause <= max_pause_duration
        fits_max = proposed_duration <= max_chunk_duration
 
        if fits_max and (below_min or short_pause):
            chunk_speech_regions.append({"start": seg_start, "end": seg_end})
            chunk_end = seg_end
        else:
            chunks.append({
                "start": chunk_start,
                "end": chunk_end,
                "speech_regions": chunk_speech_regions,
            })
            chunk_start = seg_start
            chunk_end = seg_end
            chunk_speech_regions = [{"start": seg_start, "end": seg_end}]
 
    if chunk_start is not None:
        chunks.append({
            "start": chunk_start,
            "end": chunk_end,
            "speech_regions": chunk_speech_regions,
        })
 
    # ── Post-process: rescue any chunk still under min_chunk_duration ────
    # Only happens when the main loop had to close the chunk because
    # adding the next segment would have exceeded max. Merge into the
    # smaller neighbour; min wins over max in this rare conflict.
    def _merge(target, source):
        target["end"] = source["end"]
        target["speech_regions"].extend(source["speech_regions"])
 
    i = 0
    while i < len(chunks):
        dur = chunks[i]["end"] - chunks[i]["start"]
        if dur >= min_chunk_duration:
            i += 1
            continue
 
        has_prev = i > 0
        has_next = i + 1 < len(chunks)
 
        if not has_prev and not has_next:
            # Lone chunk shorter than min — nothing we can do
            break
 
        prev_dur = (chunks[i - 1]["end"] - chunks[i - 1]["start"]) if has_prev else float("inf")
        next_dur = (chunks[i + 1]["end"] - chunks[i + 1]["start"]) if has_next else float("inf")
 
        # Prefer the smaller neighbour to keep durations balanced
        if has_prev and (not has_next or prev_dur <= next_dur):
            _merge(chunks[i - 1], chunks[i])
            chunks.pop(i)
            i = max(0, i - 1)   # re-check merged chunk
        else:
            _merge(chunks[i], chunks[i + 1])
            chunks.pop(i + 1)
            # keep i — re-check merged chunk
 
    # Add silence regions + padding
    audio_duration = len(wav) / sampling_rate
    for chunk in chunks:
        speech_regions = chunk["speech_regions"]
 
        silence_regions = []
        for i in range(len(speech_regions) - 1):
            s1 = speech_regions[i]["end"]
            s2 = speech_regions[i + 1]["start"]
            if s2 > s1:
                silence_regions.append({"start": s1, "end": s2})
 
        chunk["silence_regions"] = silence_regions
        chunk["pad_start"] = max(0.0, chunk["start"] - overlap)
        chunk["pad_end"] = min(audio_duration, chunk["end"] + overlap)
 
    return chunks


In [28]:
import math
import torch

# Assumed already importable in your project:
#   from rVADfast import rVADfast
#   from your_utils import vad_to_speech_ts


# ─────────────────────────────────────────────────────────────────────────
# Helpers for splitting overlong VAD segments on internal pauses
# ─────────────────────────────────────────────────────────────────────────

def _find_pauses(audio, sampling_rate, min_pause=0.1, energy_percentile=15):
    """
    Find quiet regions inside `audio` (1D tensor).
    A 'pause' is a run of low-energy frames lasting at least `min_pause`
    seconds. Returns list of {start, end, mid, duration} in seconds,
    local to `audio`.
    """
    frame_len = int(0.02 * sampling_rate)        # 20 ms frames
    n_frames = audio.numel() // frame_len
    if n_frames == 0:
        return []

    frames = audio[: n_frames * frame_len].reshape(n_frames, frame_len)
    energy = frames.pow(2).mean(dim=1).sqrt()

    # Adaptive threshold: bottom Nth percentile of this segment's own energy.
    # Robust to overall loudness; assumes some frames are quieter than others.
    threshold = torch.quantile(energy, energy_percentile / 100.0)
    is_quiet = (energy < threshold).tolist()

    pauses = []
    min_pause_frames = max(1, int(min_pause * sampling_rate / frame_len))

    i = 0
    while i < len(is_quiet):
        if not is_quiet[i]:
            i += 1
            continue
        j = i
        while j < len(is_quiet) and is_quiet[j]:
            j += 1
        if j - i >= min_pause_frames:
            start_t = i * frame_len / sampling_rate
            end_t = j * frame_len / sampling_rate
            pauses.append({
                "start": start_t,
                "end": end_t,
                "mid": 0.5 * (start_t + end_t),
                "duration": end_t - start_t,
            })
        i = j

    return pauses


def _split_long_segment(
    seg,
    wav,
    sampling_rate,
    max_chunk_duration,
    min_chunk_duration,
    min_pause=0.1,
    energy_percentile=15,
):
    """
    Split a VAD segment longer than max_chunk_duration on its quietest
    internal pauses. Returns a list of {start, end} pieces.

    Decision tiers (in order):
      1) Pause inside [cur+min, cur+max] -> pick LONGEST (natural cut)
      2) Any pause in (cur, cur+max]     -> pick LATEST  (max piece size)
      3) No pause within max range       -> earliest future pause
                                            (piece may exceed max;
                                             silence-only wins over cap)
    """
    duration = seg["end"] - seg["start"]
    if duration <= max_chunk_duration:
        return [seg]

    # Pull the audio for this segment, find local pauses, lift to absolute time
    s_idx = int(seg["start"] * sampling_rate)
    e_idx = int(seg["end"] * sampling_rate)
    local_pauses = _find_pauses(
        wav[s_idx:e_idx], sampling_rate, min_pause, energy_percentile,
    )
    pauses = [{
        "start":    seg["start"] + p["start"],
        "end":      seg["start"] + p["end"],
        "mid":      seg["start"] + p["mid"],
        "duration": p["duration"],
    } for p in local_pauses]

    split_points = [seg["start"]]
    cur = seg["start"]

    while seg["end"] - cur > max_chunk_duration:
        # 1) Ideal: pause inside [cur+min, cur+max]
        ideal = [p for p in pauses
                 if cur + min_chunk_duration <= p["mid"] <= cur + max_chunk_duration]
        if ideal:
            best = max(ideal, key=lambda p: (p["duration"], p["mid"]))

        else:
            # 2) Any pause within max (rescue pass will fix sub-min pieces)
            upto_max = [p for p in pauses
                        if cur < p["mid"] <= cur + max_chunk_duration]
            if upto_max:
                best = max(upto_max, key=lambda p: p["mid"])

            else:
                # 3) No pause inside max — earliest future pause anyway
                future = [p for p in pauses if p["mid"] > cur]
                if not future:
                    break  # no silence at all, leave remainder as one piece
                best = min(future, key=lambda p: p["mid"])

        split_points.append(best["mid"])
        cur = best["mid"]

    split_points.append(seg["end"])
    split_points = sorted(set(split_points))

    return [{"start": split_points[i], "end": split_points[i + 1]}
            for i in range(len(split_points) - 1)]


# ─────────────────────────────────────────────────────────────────────────
# Main entry point
# ─────────────────────────────────────────────────────────────────────────

def vad_chunk_with_timestamps_v3(
    wav,
    sampling_rate=16000,
    max_chunk_duration=10.0,
    max_pause_duration=1.5,
    overlap=0.5,
    min_chunk_duration=2.0,
    split_min_pause=0.1,
    split_energy_percentile=15,
):
    """
    wav: torch.Tensor (1D, 16kHz)

    returns: list of dicts with:
        start/end         -> true speech span of the chunk
        pad_start/pad_end -> padded boundaries used for audio extraction
        speech_regions    -> list of speech intervals inside the chunk
        silence_regions   -> list of silence gaps inside the chunk

    Final chunks are kept within [min_chunk_duration, max_chunk_duration]
    whenever possible. Only unfixable case: the whole audio yields a
    single chunk shorter than min_chunk_duration (no neighbour to merge).

    Long VAD segments (>max_chunk_duration) are split on internal pauses
    detected via short-time energy. Controlled by:
        split_min_pause          - min silence duration to count as a pause
        split_energy_percentile  - per-segment energy threshold (percentile)
    """
    vad = rVADfast()
    vad_labels, vad_timestamps = vad(wav, sampling_rate)
    speech_ts = vad_to_speech_ts(vad_labels, vad_timestamps, sampling_rate)

    # Convert VAD timestamps from samples to seconds
    split_ts = []
    for seg in speech_ts:
        seg_start = seg["start"] / sampling_rate
        seg_end = seg["end"] / sampling_rate
        split_ts.append({"start": seg_start, "end": seg_end})

    # ── Split any VAD segment longer than max_chunk_duration ─────────────
    # rVADfast can produce very long continuous segments (100s+ of speech
    # with no detected silence). Cut them on their internal pauses so the
    # merging loop below can treat them like normal-sized segments.
    split_ts = [
        p
        for seg in split_ts
        for p in _split_long_segment(
            seg, wav, sampling_rate,
            max_chunk_duration, min_chunk_duration,
            min_pause=split_min_pause,
            energy_percentile=split_energy_percentile,
        )
    ]

    # ── Merge split_ts segments, enforcing min/max chunk duration ────────
    # A chunk keeps growing while EITHER:
    #   • it is still below min_chunk_duration   (force-grow), OR
    #   • the pause to the next segment is short (normal merging)
    # …provided the result still fits under max_chunk_duration.
    chunks = []
    chunk_start = None
    chunk_end = None
    chunk_speech_regions = []

    for seg in split_ts:
        seg_start = seg["start"]
        seg_end = seg["end"]

        if chunk_start is None:
            chunk_start = seg_start
            chunk_end = seg_end
            chunk_speech_regions = [{"start": seg_start, "end": seg_end}]
            continue

        current_duration = chunk_end - chunk_start
        pause = seg_start - chunk_end
        proposed_duration = seg_end - chunk_start

        below_min = current_duration < min_chunk_duration
        short_pause = pause <= max_pause_duration
        fits_max = proposed_duration <= max_chunk_duration

        if fits_max and (below_min or short_pause):
            chunk_speech_regions.append({"start": seg_start, "end": seg_end})
            chunk_end = seg_end
        else:
            chunks.append({
                "start": chunk_start,
                "end": chunk_end,
                "speech_regions": chunk_speech_regions,
            })
            chunk_start = seg_start
            chunk_end = seg_end
            chunk_speech_regions = [{"start": seg_start, "end": seg_end}]

    if chunk_start is not None:
        chunks.append({
            "start": chunk_start,
            "end": chunk_end,
            "speech_regions": chunk_speech_regions,
        })

    # ── Post-process: rescue any chunk still under min_chunk_duration ────
    # Only happens when the main loop had to close the chunk because
    # adding the next segment would have exceeded max. Merge into the
    # smaller neighbour; min wins over max in this rare conflict.
    def _merge(target, source):
        target["end"] = source["end"]
        target["speech_regions"].extend(source["speech_regions"])

    i = 0
    while i < len(chunks):
        dur = chunks[i]["end"] - chunks[i]["start"]
        if dur >= min_chunk_duration:
            i += 1
            continue

        has_prev = i > 0
        has_next = i + 1 < len(chunks)

        if not has_prev and not has_next:
            # Lone chunk shorter than min — nothing we can do
            break

        prev_dur = (chunks[i - 1]["end"] - chunks[i - 1]["start"]) if has_prev else float("inf")
        next_dur = (chunks[i + 1]["end"] - chunks[i + 1]["start"]) if has_next else float("inf")

        # Prefer the smaller neighbour to keep durations balanced
        if has_prev and (not has_next or prev_dur <= next_dur):
            _merge(chunks[i - 1], chunks[i])
            chunks.pop(i)
            i = max(0, i - 1)   # re-check merged chunk
        else:
            _merge(chunks[i], chunks[i + 1])
            chunks.pop(i + 1)
            # keep i — re-check merged chunk

    # ── Add silence regions + padding ────────────────────────────────────
    audio_duration = len(wav) / sampling_rate
    for chunk in chunks:
        speech_regions = chunk["speech_regions"]

        silence_regions = []
        for i in range(len(speech_regions) - 1):
            s1 = speech_regions[i]["end"]
            s2 = speech_regions[i + 1]["start"]
            if s2 > s1:
                silence_regions.append({"start": s1, "end": s2})

        chunk["silence_regions"] = silence_regions
        chunk["pad_start"] = max(0.0, chunk["start"] - overlap)
        chunk["pad_end"] = min(audio_duration, chunk["end"] + overlap)

    return chunks

In [48]:
"""from VAD_chunk import vad_chunk_with_timestamps_v3
import os
import shutil
audio_files = sorted(
    glob.glob(os.path.join(audio_dir, "**", "*.wav"), recursive=True)
)
directory="w2vCTC_softpool_xphonebert"
for checkpoint in os.listdir(f"results/{directory}"):
    #checkpoint = "results/w2vCTC_joint_nofxfy/checkpoint-12499"
    if checkpoint.startswith("tensorboard"):
        continue
    print(checkpoint)
    checkpoint_path = f"results/{directory}/{checkpoint}"
    bin_path = os.path.join(checkpoint_path, "pytorch_model.bin")
    sft_path = os.path.join(checkpoint_path, "model.safetensors")
    if os.path.exists(bin_path):
        state_dict = torch.load(bin_path, map_location="cpu")
    elif os.path.exists(sft_path):
        from safetensors.torch import load_file
        state_dict = load_file(sft_path)
    else:
        raise FileNotFoundError(
            f"No model weights found in {checkpoint}\n"
            f"Expected pytorch_model.bin or model.safetensors"
        )
    
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    
    model.eval()
    model.to(device)
    print("Model ready.")
    results = {}
    n_done  = 0
    
    for audio_path in os.listdir(audio_dir):
        audio_path = audio_dir + audio_path
        filename   = os.path.splitext(os.path.basename(audio_path))[0]
    
        if filename == "Rhap-D2004":
            continue
    
        tg_path = os.path.join(textgrid_dir, filename + "-Pro.TextGrid")
        if not os.path.exists(tg_path):
            print(f"  [SKIP] No TextGrid for {filename}")
            continue
    
        clean_ref     = read_textgrid(tg_path, tier)
        ref_intervals = get_ref_intervals(clean_ref, audio_path)
    
        audio, sr = sf.read(audio_path)
        if sr != 16000:
            import librosa
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
    
        wav    = torch.from_numpy(audio.astype(np.float32))
        chunks = vad_chunk_with_timestamps_v3(wav)
        for i, c in enumerate(chunks):
            dur = c["end"] - c["start"]
            print(f"  chunk {i}: {c['start']:.2f}→{c['end']:.2f}  duration={dur:.2f}s")
            if dur > 30.0:
                print(f"    ⚠ chunk {i} exceeds 30s!")
        audio_duration = len(audio) / 16000
        
        covered = sum(c["end"] - c["start"] for c in chunks) if chunks else 0
       
        spn_count = sum(1 for iv in ref_intervals if iv["phoneme"] == "_")
        # Fix 1: fallback chunk must include pad_start/pad_end
        if not chunks:
            duration = len(audio) / 16000
            chunks = [{
                "start":     0.0,
                "end":       duration,
                "pad_start": 0.0,
                "pad_end":   duration,
            }]
    
        hyp_intervals = []
        a_matrices    = []
        all_logits    = []
    
        for chunk_idx, chunk in enumerate(chunks):
            start_sample = int(chunk["pad_start"] * 16000)
            end_sample   = int(chunk["pad_end"]   * 16000)
            chunk_audio  = audio[start_sample:end_sample]
            if len(chunk_audio) < 400:
                continue
    
            inputs = feature_extractor(
                chunk_audio, sampling_rate=16000,
                return_tensors="pt", return_attention_mask=True,
            )
    
            with torch.no_grad():
                outputs = model(
                    inputs.input_values.to(device),
                    attention_mask=inputs.attention_mask.to(device),
                )
                logits = outputs["logits"]                # [B, T, V]
                A = outputs["alignment"]         # [B, P, T]
    
            dur = len(chunk_audio) / 16000.0
            ivs = extract_intervals_forced_vad(
                logits, A, tokenizer, dur,
                chunk=chunk,
                plot_path=None,
                plot_title="alignment-matrix",
            )
            true_start = chunk["start"]
            true_end   = chunk["end"]

    
            kept = 0
            for iv in ivs:
                abs_start = round(iv["start"] + chunk["pad_start"], 6)
                abs_end   = round(iv["end"]   + chunk["pad_start"], 6)
                centre    = (abs_start + abs_end) / 2.0
                if not (true_start - 0.01 <= centre <= true_end + 0.01):
                    continue
                hyp_intervals.append({
                    "phoneme": normalise_hyp_token(iv["phoneme"]),
                    "start":   abs_start,
                    "end":     abs_end,
                })
                kept += 1
    
            a_matrices.append(A[0].cpu())
            all_logits.append(logits.cpu())
    
        hyp_intervals.sort(key=lambda iv: iv["start"])
    
    
        total_ctc_phones = sum(
        len([t for t in logits[0].argmax(-1).tolist()
             if t != tokenizer.pad_token_id])  # raw non-blank count
        for logits in all_logits)
        print(f"  Raw CTC non-blank frames: {total_ctc_phones}, "
          f"hyp intervals: {len(hyp_intervals)}, "
          f"ref: {len(ref_intervals)}")
        results[filename] = {
            "file":          os.path.basename(audio_path),
            "ref_intervals": ref_intervals,
            "hyp_intervals": hyp_intervals,
            "A_chunks":      a_matrices,
            "logits_chunks": all_logits,
        }
    
        n_done += 1
        print(f"[{n_done}] Done: {filename}  "
              f"({len(hyp_intervals)} hyp intervals, {len(ref_intervals)} ref intervals)")
    with open(f"results/{directory}_{checkpoint}.pkl", "wb") as f:
        pickle.dump(results, f)
    for fname, entry in results.items():
        if os.path.exists(f"results/{directory}-{checkpoint}/{fname}-align"):
            shutil.rmtree(f"results/{directory}-{checkpoint}/{fname}-align")
        os.makedirs(f"results/{directory}-{checkpoint}/{fname}-align")
        visualize_alignment_with_dtw(entry, fname, tokenizer,out_dir=f"results/{directory}-{checkpoint}/{fname}-align")
    alignment_store = prepare_alignment_store(results)
    with open(f"results/{directory}-{checkpoint}/alignment_dict.pkl", "wb") as f:
        pickle.dump(alignment_store, f)
    metrics(alignment_store, f"results/{directory}-{checkpoint}/metrics_per_file.csv")
    #pkl_to_etf_aligned(f"results/{directory}-{checkpoint}/alignment_dict.pkl",vocab, f"results/{directory}-{checkpoint}/ref.etf",f"results/{directory}-{checkpoint}/hyp.etf")
"""

'from VAD_chunk import vad_chunk_with_timestamps_v3\nimport os\nimport shutil\naudio_files = sorted(\n    glob.glob(os.path.join(audio_dir, "**", "*.wav"), recursive=True)\n)\ndirectory="w2vCTC_softpool_xphonebert"\nfor checkpoint in os.listdir(f"results/{directory}"):\n    #checkpoint = "results/w2vCTC_joint_nofxfy/checkpoint-12499"\n    if checkpoint.startswith("tensorboard"):\n        continue\n    print(checkpoint)\n    checkpoint_path = f"results/{directory}/{checkpoint}"\n    bin_path = os.path.join(checkpoint_path, "pytorch_model.bin")\n    sft_path = os.path.join(checkpoint_path, "model.safetensors")\n    if os.path.exists(bin_path):\n        state_dict = torch.load(bin_path, map_location="cpu")\n    elif os.path.exists(sft_path):\n        from safetensors.torch import load_file\n        state_dict = load_file(sft_path)\n    else:\n        raise FileNotFoundError(\n            f"No model weights found in {checkpoint}\n"\n            f"Expected pytorch_model.bin or model.safeten

In [16]:
import torch


def vad_chunk_with_timestamps_v4(
    wav,
    sampling_rate=16000,
    max_chunk_duration=10.0,
    max_pause_duration=1.5,
    overlap=0.5,
):
    """
    wav: torch.Tensor (1D, 16kHz)

    returns: list of dicts with:
        start/end         -> true speech span of the chunk
        pad_start/pad_end -> padded boundaries used for audio extraction
        speech_regions    -> list of speech intervals inside the chunk
        silence_regions   -> list of silence gaps inside the chunk

    Behavior:
        - merge adjacent speech segments if pause <= max_pause_duration
          and merged duration <= max_chunk_duration
        - if a resulting chunk is > max_chunk_duration, split it at the
          largest internal silence
        - if no internal silence exists, keep the chunk as-is
    """

    vad = rVADfast()
    vad_labels, vad_timestamps = vad(wav, sampling_rate)
    speech_ts = vad_to_speech_ts(vad_labels, vad_timestamps, 16000)

    # Convert VAD timestamps from samples to seconds
    split_ts = []
    for seg in speech_ts:
        split_ts.append({
            "start": seg["start"] / sampling_rate,
            "end": seg["end"] / sampling_rate,
        })

    def chunk_duration(regions):
        return regions[-1]["end"] - regions[0]["start"]

    def build_chunk_from_regions(regions):
        return {
            "start": regions[0]["start"],
            "end": regions[-1]["end"],
            "speech_regions": regions,
        }

    def split_chunk_at_largest_silence(regions):
        """
        Split a chunk into two parts at the largest silence gap.
        Returns (left_regions, right_regions), or None if no silence gap exists.
        """
        if len(regions) <= 1:
            return None

        best_i = None
        best_gap = 0.0

        for i in range(len(regions) - 1):
            gap = regions[i + 1]["start"] - regions[i]["end"]
            if gap > best_gap:
                best_gap = gap
                best_i = i

        if best_i is None or best_gap <= 0:
            return None

        left_regions = regions[: best_i + 1]
        right_regions = regions[best_i + 1 :]
        return left_regions, right_regions

    def enforce_soft_max_duration(chunk):
        """
        If the chunk is longer than max_chunk_duration, split it at the largest
        internal silence. If there is no silence, keep it long.
        """
        regions = chunk["speech_regions"]

        if chunk_duration(regions) <= max_chunk_duration:
            return [chunk]

        split = split_chunk_at_largest_silence(regions)
        if split is None:
            # No silence inside -> keep the chunk long
            return [chunk]

        left_regions, right_regions = split
        left_chunk = build_chunk_from_regions(left_regions)
        right_chunk = build_chunk_from_regions(right_regions)

        return enforce_soft_max_duration(left_chunk) + enforce_soft_max_duration(right_chunk)

    # First pass: greedy grouping
    raw_chunks = []
    current_regions = []

    for seg in split_ts:
        if not current_regions:
            current_regions = [seg]
            continue

        current_start = current_regions[0]["start"]
        current_end = current_regions[-1]["end"]

        pause = seg["start"] - current_end
        proposed_duration = seg["end"] - current_start

        if pause <= max_pause_duration and proposed_duration <= max_chunk_duration:
            current_regions.append(seg)
        else:
            raw_chunks.append(build_chunk_from_regions(current_regions))
            current_regions = [seg]

    if current_regions:
        raw_chunks.append(build_chunk_from_regions(current_regions))

    # Second pass: split oversized chunks only at internal silence
    chunks = []
    for chunk in raw_chunks:
        chunks.extend(enforce_soft_max_duration(chunk))

    # Add silence regions + padding
    audio_duration = len(wav) / sampling_rate
    for chunk in chunks:
        speech_regions = chunk["speech_regions"]

        silence_regions = []
        for i in range(len(speech_regions) - 1):
            s1 = speech_regions[i]["end"]
            s2 = speech_regions[i + 1]["start"]
            if s2 > s1:
                silence_regions.append({"start": s1, "end": s2})

        chunk["silence_regions"] = silence_regions
        chunk["pad_start"] = max(0.0, chunk["start"] - overlap)
        chunk["pad_end"] = min(audio_duration, chunk["end"] + overlap)

    return chunks

In [20]:
def vad_chunk_with_timestamps_v3(
    wav,
    sampling_rate=16000,
    max_chunk_duration=10.0,
    max_pause_duration=1.5,
    overlap=0.5,
):
    """
    wav: torch.Tensor (1D, 16kHz)

    returns: list of dicts with:
        start/end         -> true speech span of the chunk
        pad_start/pad_end -> padded boundaries used for audio extraction
        speech_regions    -> list of speech intervals inside the chunk
        silence_regions   -> list of silence gaps inside the chunk
    """
    vad = rVADfast()
    vad_labels, vad_timestamps = vad(wav, sampling_rate)
    speech_ts = vad_to_speech_ts(vad_labels, vad_timestamps, sampling_rate)

    # Convert VAD timestamps from samples to seconds
    split_ts = []
    for seg in speech_ts:
        seg_start = seg["start"] / sampling_rate
        seg_end = seg["end"] / sampling_rate
        split_ts.append({"start": seg_start, "end": seg_end})

    chunks = []
    chunk_start = None
    chunk_end = None
    chunk_speech_regions = []

    for seg in split_ts:
        seg_start = seg["start"]
        seg_end = seg["end"]

        if chunk_start is None:
            chunk_start = seg_start
            chunk_end = seg_end
            chunk_speech_regions = [{"start": seg_start, "end": seg_end}]
            continue

        pause = seg_start - chunk_end
        proposed_duration = seg_end - chunk_start

        if pause <= max_pause_duration and proposed_duration <= max_chunk_duration:
            chunk_speech_regions.append({"start": seg_start, "end": seg_end})
            chunk_end = seg_end
        else:
            chunks.append({
                "start": chunk_start,
                "end": chunk_end,
                "speech_regions": chunk_speech_regions,
            })
            chunk_start = seg_start
            chunk_end = seg_end
            chunk_speech_regions = [{"start": seg_start, "end": seg_end}]

    if chunk_start is not None:
        chunks.append({
            "start": chunk_start,
            "end": chunk_end,
            "speech_regions": chunk_speech_regions,
        })

    # Add silence regions + padding
    audio_duration = len(wav) / sampling_rate
    for chunk in chunks:
        speech_regions = chunk["speech_regions"]

        silence_regions = []
        for i in range(len(speech_regions) - 1):
            s1 = speech_regions[i]["end"]
            s2 = speech_regions[i + 1]["start"]
            if s2 > s1:
                silence_regions.append({"start": s1, "end": s2})

        chunk["silence_regions"] = silence_regions
        chunk["pad_start"] = max(0.0, chunk["start"] - overlap)
        chunk["pad_end"] = min(audio_duration, chunk["end"] + overlap)

    return chunks

In [24]:
from VAD_chunk import vad_chunk_with_timestamps_v3
import os
import math
import shutil
audio_files = sorted(
    glob.glob(os.path.join(audio_dir, "**", "*.wav"), recursive=True)
)
directory="w2vCTC_softpool_xphonebert"
checkpoint ="checkpoint-12498"
checkpoint_path = f"results/{directory}/{checkpoint}"
bin_path = os.path.join(checkpoint_path, "pytorch_model.bin")
sft_path = os.path.join(checkpoint_path, "model.safetensors")
if os.path.exists(bin_path):
    state_dict = torch.load(bin_path, map_location="cpu")
elif os.path.exists(sft_path):
    from safetensors.torch import load_file
    state_dict = load_file(sft_path)
else:
    raise FileNotFoundError(
        f"No model weights found in {checkpoint}\n"
        f"Expected pytorch_model.bin or model.safetensors"
    )

missing, unexpected = model.load_state_dict(state_dict, strict=False)

model.eval()
model.to(device)
print("Model ready.")
results = {}
n_done  = 0

for audio_path in os.listdir(audio_dir):
    audio_path = audio_dir + audio_path
    filename   = os.path.splitext(os.path.basename(audio_path))[0]

    if filename == "Rhap-D2004":
        continue

    tg_path = os.path.join(textgrid_dir, filename + "-Pro.TextGrid")
    if not os.path.exists(tg_path):
        print(f"  [SKIP] No TextGrid for {filename}")
        continue

    clean_ref     = read_textgrid(tg_path, tier)
    ref_intervals = get_ref_intervals(clean_ref, audio_path)

    audio, sr = sf.read(audio_path)
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    wav    = torch.from_numpy(audio.astype(np.float32))
    chunks = vad_chunk_with_timestamps_v3(wav)
    for i, c in enumerate(chunks):
        dur = c["end"] - c["start"]
        print(f"  chunk {i}: {c['start']:.2f}→{c['end']:.2f}  duration={dur:.2f}s")
        if dur > 10.0:
            print(f"    ⚠ chunk {i} exceeds 30s!")
    audio_duration = len(audio) / 16000
    
    covered = sum(c["end"] - c["start"] for c in chunks) if chunks else 0
   
    spn_count = sum(1 for iv in ref_intervals if iv["phoneme"] == "_")
    # Fix 1: fallback chunk must include pad_start/pad_end
    if not chunks:
        duration = len(audio) / 16000
        chunks = [{
            "start":     0.0,
            "end":       duration,
            "pad_start": 0.0,
            "pad_end":   duration,
        }]

    hyp_intervals = []
    a_matrices    = []
    all_logits    = []

    for chunk_idx, chunk in enumerate(chunks):
        start_sample = int(chunk["pad_start"] * 16000)
        end_sample   = int(chunk["pad_end"]   * 16000)
        chunk_audio  = audio[start_sample:end_sample]
        if len(chunk_audio) < 400:
            continue

        inputs = feature_extractor(
            chunk_audio, sampling_rate=16000,
            return_tensors="pt", return_attention_mask=True,
        )

        with torch.no_grad():
            outputs = model(
                inputs.input_values.to(device),
                attention_mask=inputs.attention_mask.to(device),
            )
            logits = outputs["logits"]                # [B, T, V]
            A = outputs["alignment"]         # [B, P, T]

        dur = len(chunk_audio) / 16000.0
        ivs = extract_intervals_forced_vad(
            logits, A, tokenizer, dur,
            chunk=chunk,
            plot_path=None,
            plot_title="alignment-matrix",
        )
        true_start = chunk["start"]
        true_end   = chunk["end"]


        kept = 0
        for iv in ivs:
            abs_start = round(iv["start"] + chunk["pad_start"], 6)
            abs_end   = round(iv["end"]   + chunk["pad_start"], 6)
            centre    = (abs_start + abs_end) / 2.0
            if not (true_start - 0.01 <= centre <= true_end + 0.01):
                continue
            hyp_intervals.append({
                "phoneme": normalise_hyp_token(iv["phoneme"]),
                "start":   abs_start,
                "end":     abs_end,
            })
            kept += 1

        a_matrices.append(A[0].cpu())
        all_logits.append(logits.cpu())

    hyp_intervals.sort(key=lambda iv: iv["start"])


    total_ctc_phones = sum(
    len([t for t in logits[0].argmax(-1).tolist()
         if t != tokenizer.pad_token_id])  # raw non-blank count
    for logits in all_logits)
    print(f"  Raw CTC non-blank frames: {total_ctc_phones}, "
      f"hyp intervals: {len(hyp_intervals)}, "
      f"ref: {len(ref_intervals)}")
    results[filename] = {
        "file":          os.path.basename(audio_path),
        "ref_intervals": ref_intervals,
        "hyp_intervals": hyp_intervals,
        "A_chunks":      a_matrices,
        "logits_chunks": all_logits,
    }

    n_done += 1
    print(f"[{n_done}] Done: {filename}  "
          f"({len(hyp_intervals)} hyp intervals, {len(ref_intervals)} ref intervals)")
with open(f"results/{directory}_{checkpoint}.pkl", "wb") as f:
    pickle.dump(results, f)
for fname, entry in results.items():
    if os.path.exists(f"results/{directory}-{checkpoint}/{fname}-align"):
        shutil.rmtree(f"results/{directory}-{checkpoint}/{fname}-align")
    os.makedirs(f"results/{directory}-{checkpoint}/{fname}-align")
    visualize_alignment_with_dtw(entry, fname, tokenizer,out_dir=f"results/{directory}-{checkpoint}/{fname}-align")
alignment_store = prepare_alignment_store(results)
with open(f"results/{directory}-{checkpoint}/alignment_dict.pkl", "wb") as f:
    pickle.dump(alignment_store, f)
metrics(alignment_store, f"results/{directory}-{checkpoint}/metrics_per_file.csv")
    #pkl_to_etf_aligned(f"results/{directory}-{checkpoint}/alignment_dict.pkl",vocab, f"results/{directory}-{checkpoint}/ref.etf",f"results/{directory}-{checkpoint}/hyp.etf")


Model ready.
  Session offset corrected (end_overshoot): ref_first=0.707s, ref_last=17.723s, audio=17.016s, end_overshoot=+0.707s, offset=0.707s


Using cache found in /home/imbenamor/.cache/torch/hub/snakers4_silero-vad_master


  chunk 0: 0.03→8.38  duration=8.35s
  chunk 1: 8.77→17.02  duration=8.25s
  Raw CTC non-blank frames: 219, hyp intervals: 164, ref: 166
[1] Done: Rhap-M0012  (164 hyp intervals, 166 ref intervals)
  Session offset corrected (end_overshoot): ref_first=0.120s, ref_last=25.938s, audio=25.817s, end_overshoot=+0.120s, offset=0.120s


Using cache found in /home/imbenamor/.cache/torch/hub/snakers4_silero-vad_master


  chunk 0: 0.00→22.17  duration=22.17s
    ⚠ chunk 0 exceeds 30s!
  chunk 1: 22.59→25.82  duration=3.22s
  Raw CTC non-blank frames: 310, hyp intervals: 234, ref: 243
[2] Done: Rhap-M0015  (234 hyp intervals, 243 ref intervals)
  Session offset corrected (end_overshoot): ref_first=4.010s, ref_last=431.895s, audio=427.885s, end_overshoot=+4.010s, offset=4.010s


...

  chunk 0: 0.00→44.34  duration=44.34s
    ⚠ chunk 0 exceeds 30s!
  Raw CTC non-blank frames: 534, hyp intervals: 377, ref: 382
[52] Done: Rhap-M0024  (377 hyp intervals, 382 ref intervals)


Using cache found in /home/imbenamor/.cache/torch/hub/snakers4_silero-vad_master


  chunk 0: 0.00→11.33  duration=11.33s
    ⚠ chunk 0 exceeds 30s!
  chunk 1: 12.16→40.22  duration=28.06s
    ⚠ chunk 1 exceeds 30s!
  chunk 2: 40.61→63.33  duration=22.72s
    ⚠ chunk 2 exceeds 30s!
  chunk 3: 63.71→66.91  duration=3.20s
  chunk 4: 67.23→111.77  duration=44.54s
    ⚠ chunk 4 exceeds 30s!
  chunk 5: 112.13→136.89  duration=24.76s
    ⚠ chunk 5 exceeds 30s!
  chunk 6: 137.60→196.61  duration=59.00s
    ⚠ chunk 6 exceeds 30s!
  chunk 7: 197.47→279.33  duration=81.85s
    ⚠ chunk 7 exceeds 30s!
  chunk 8: 279.62→284.38  duration=4.76s
  chunk 9: 284.93→304.77  duration=19.84s
    ⚠ chunk 9 exceeds 30s!
  chunk 10: 305.06→357.09  duration=52.03s
    ⚠ chunk 10 exceeds 30s!
  chunk 11: 357.51→385.58  duration=28.07s
    ⚠ chunk 11 exceeds 30s!
  Raw CTC non-blank frames: 5461, hyp intervals: 3800, ref: 3811
[53] Done: Rhap-M2002  (3800 hyp intervals, 3811 ref intervals)


/tmp/ipykernel_431141/4191958114.py:90: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved: results/w2vCTC_softpool_xphonebert-checkpoint-12498/Rhap-M0012-align/Rhap-M0012_chunk01.png
  Saved: results/w2vCTC_softpool_xphonebert-checkpoint-12498/Rhap-M0012-align/Rhap-M0012_chunk02.png
  Saved: results/w2vCTC_softpool_xphonebert-checkpoint-12498/Rhap-M0015-align/Rhap-M0015_chunk01.png
  Saved: results/w2vCTC_softpool_xphonebert-checkpoint-12498/Rhap-M0015-align/Rhap-M0015_chunk02.png
  Saved: results/w2vCTC_softpool_xphonebert-checkpoint-12498/Rhap-D1001-align/Rhap-D1001_chunk01.png
  Saved: results/w2vCTC_softpool_xphonebert-checkpoint-12498/Rhap-D1001-align/Rhap-D1001_chunk02.png
  Saved: results/w2vCTC_softpool_xphonebert-checkpoint-12498/Rhap-D1001-align/Rhap-D1001_chunk03.png
  Saved: results/w2vCTC_softpool_xphonebert-checkpoint-12498/Rhap-D1001-align/Rhap-D1001_chunk04.png
  Saved: results/w2vCTC_softpool_xphonebert-checkpoint-12498/Rhap-D1001-align/Rhap-D1001_chunk05.png
  Saved: results/w2vCTC_softpool_xphonebert-checkpoint-12498/Rhap-D1001-align/Rhap-D1001_ch

In [11]:
from VAD_chunk import vad_chunk_with_timestamps_v3


audio_files = sorted(
    glob.glob(os.path.join(audio_dir, "**", "*.wav"), recursive=True)
)
directory="w2vCTC_bertphone"
checkpoint = "checkpoint-14841"

checkpoint_path = f"results/{directory}/{checkpoint}"
bin_path = os.path.join(checkpoint_path, "pytorch_model.bin")
sft_path = os.path.join(checkpoint_path, "model.safetensors")
if os.path.exists(bin_path):
    state_dict = torch.load(bin_path, map_location="cpu")
elif os.path.exists(sft_path):
    from safetensors.torch import load_file
    state_dict = load_file(sft_path)
else:
    raise FileNotFoundError(
        f"No model weights found in {checkpoint}\n"
        f"Expected pytorch_model.bin or model.safetensors"
    )

missing, unexpected = model.load_state_dict(state_dict, strict=False)

model.eval()
model.to(device)
print("Model ready.")
results = {}
n_done  = 0

for audio_path in sorted(os.listdir(audio_dir)):
    audio_path = audio_dir + audio_path
    filename   = os.path.splitext(os.path.basename(audio_path))[0]
    print(filename)
    if filename == "Rhap-D2004" or filename =="Rhap-M2005" or filename=="Rhap-D2008":
        continue

    tg_path = os.path.join(textgrid_dir, filename + "-Pro.TextGrid")
    if not os.path.exists(tg_path):
        print(f"  [SKIP] No TextGrid for {filename}")
        continue

    clean_ref     = read_textgrid(tg_path, tier)
    ref_intervals = get_ref_intervals(clean_ref, audio_path)

    audio, sr = sf.read(audio_path)
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    wav    = torch.from_numpy(audio.astype(np.float32))
    chunks = vad_chunk_with_timestamps_v3(wav)
    audio_duration = len(audio) / 16000
    covered = sum(c["end"] - c["start"] for c in chunks) if chunks else 0
    print(f"  VAD coverage: {covered:.1f}s / {audio_duration:.1f}s "
          f"({100*covered/audio_duration:.1f}%)")

    spn_count = sum(1 for iv in ref_intervals if iv["phoneme"] == "_")
    print(f"  Ref spn count: {spn_count} / {len(ref_intervals)}")
    # Fix 1: fallback chunk must include pad_start/pad_end
    if not chunks:
        duration = len(audio) / 16000
        chunks = [{
            "start":     0.0,
            "end":       duration,
            "pad_start": 0.0,
            "pad_end":   duration,
        }]

    hyp_intervals = []
    a_matrices    = []
    all_logits    = []

    for chunk_idx, chunk in enumerate(chunks):
        start_sample = int(chunk["pad_start"] * 16000)
        end_sample   = int(chunk["pad_end"]   * 16000)
        chunk_audio  = audio[start_sample:end_sample]

        if len(chunk_audio) < 400:
            continue

        inputs = feature_extractor(
            chunk_audio, sampling_rate=16000,
            return_tensors="pt", return_attention_mask=True,
        )

        with torch.no_grad():
            outputs = model(
                inputs.input_values.to(device),
                attention_mask=inputs.attention_mask.to(device),
            )
            logits = outputs["logits"]                # [B, T, V]
            A = outputs["alignment"]         # [B, P, T]

        dur = len(chunk_audio) / 16000.0
        ivs = extract_intervals_forced_vad(
            logits, A, tokenizer, dur,
            chunk=chunk,
            plot_path=None,
            plot_title="alignment-matrix",
        )
        true_start = chunk["start"]
        true_end   = chunk["end"]
        print(f"  chunk[{chunk_idx}]: "
              f"pad=[{chunk['pad_start']:.2f}, {chunk['pad_end']:.2f}]  "
              f"true=[{true_start:.2f}, {true_end:.2f}]  "
              f"n_ivs={len(ivs)}")

        kept = 0
        for iv in ivs:
            abs_start = round(iv["start"] + chunk["pad_start"], 6)
            abs_end   = round(iv["end"]   + chunk["pad_start"], 6)
            centre    = (abs_start + abs_end) / 2.0
            if not (true_start - 0.01 <= centre <= true_end + 0.01):
                continue
            hyp_intervals.append({
                "phoneme": normalise_hyp_token(iv["phoneme"]),
                "start":   abs_start,
                "end":     abs_end,
            })
            kept += 1
        print(f"    kept (true-window centre filter): {kept}")

        a_matrices.append(A[0].cpu())
        all_logits.append(logits.cpu())

    hyp_intervals.sort(key=lambda iv: iv["start"])


    total_ctc_phones = sum(
    len([t for t in logits[0].argmax(-1).tolist()
         if t != tokenizer.pad_token_id])  # raw non-blank count
    for logits in all_logits)
    print(f"  Raw CTC non-blank frames: {total_ctc_phones}, "
      f"hyp intervals: {len(hyp_intervals)}, "
      f"ref: {len(ref_intervals)}")
    results[filename] = {
        "file":          os.path.basename(audio_path),
        "ref_intervals": ref_intervals,
        "hyp_intervals": hyp_intervals,
        "A_chunks":      a_matrices,
        "logits_chunks": all_logits,
    }

for fname, entry in results.items():
    os.makedirs(f"results/{directory}-{checkpoint}/{fname}-align")
    visualize_alignment_with_dtw(entry, fname, tokenizer,out_dir=f"results/{directory}-{checkpoint}/{fname}-align")
alignment_store = prepare_alignment_store(results)
#with open(f"results/{directory}-{checkpoint}/alignment_dict.pkl", "wb") as f:
    #pickle.dump(alignment_store, f)
metrics(alignment_store, f"results/{directory}-{checkpoint}/metrics_per_file.csv")
#pkl_to_etf_aligned(f"results/{directory}-{checkpoint}/alignment_dict.pkl",vocab, f"results/{directory}-{checkpoint}/ref.etf",f"results/{directory}-{checkpoint}/hyp.etf")


KeyboardInterrupt: 